# Create the LePHARE 'model' and submission files
In this notebook we perform the setup required to run LePHARE on the test data. Currently it runs all the combinations of tasksets, sims, scenarios and test and training samples to allow validation.

The insturctions for the challenge are here:

https://pz-data-challenge.readthedocs.io/en/latest/#

The predictions from the test are uploaded as a tar file. We also specify a test_lephare.py file which directly runs lephare on the test data.

When all is ready to submit we push a branch submit_lephare to the main github page for the challenge:

https://github.com/LSSTDESC/pz_data_challenge

In [ ]:
import os
import numpy as np
from rail.estimation.algos.lephare import LephareInformer, LephareEstimator, lsst_default_config
import lephare as lp
import tables_io
from astropy.table import Table
import qp
from matplotlib import pylab as plt

In [ ]:
# To get the data in public you must run ../scripts/download_public.py
!ls ../public

## Load all the training and testing samples

In [ ]:
TASKSETS = ["taskset_1", "taskset_2", "taskset_3", "taskset_4"]
SIMS = ["cardinal", "flagship"]
SCENARIOS = ["1yr", "10yr"]
DATADIR = '../public'
SUBMISSION_DIR = '../evaluation'
data_dict = {}
for taskset_ in TASKSETS:
    for sim_ in SIMS:
        for scenario_ in SCENARIOS:
            for run_ in ['training','test']:
                key = f"{taskset_}_{sim_}_{run_}_{scenario_}"
                file = os.path.abspath(os.path.join(DATADIR, f"pz_challenge_{key}.hdf5"))
                data_dict[f"{key}"] = tables_io.read(file)


In [ ]:
data_dict.keys()

In [ ]:

# We currently only train once so don't need to rerun for every task
train_data_handle  = tables_io.read('../public/pz_challenge_taskset_2_cardinal_training_10yr.hdf5')

In [ ]:
train_data_handle.keys()

In [ ]:
train_data_handle['mag_i_lsst']

In [ ]:
plt.scatter(train_data_handle['ra'],train_data_handle['dec'])

In [ ]:
plt.scatter(train_data_handle['ra'],train_data_handle['dec'])

In [ ]:
np.sum(~np.isfinite(train_data_handle['mag_H_roman_err']))

In [ ]:
plt.hist(train_data_handle['mag_H_roman_err'],bins=30)
# plt.yscale('log')

In [ ]:
plt.hist(train_data_handle['mag_i_lsst'],bins=30)
plt.yscale('log')

In [ ]:
plt.scatter(train_data_handle['redshift'],train_data_handle['mag_i_lsst']-train_data_handle['mag_z_lsst'],s=0.01)

In [ ]:
plt.hist(train_data_handle['mag_r_lsst'],bins=30)
plt.yscale('log')

In [ ]:
plt.hist(train_data_handle['redshift'],bins=30)

In [ ]:
flux_cols=[f'mag_{b}_lsst' for b in 'ugrizy']
flux_cols+=[f'mag_{b}_roman' for b in 'YJH']
flux_err_cols=[f'mag_{b}_lsst_err' for b in 'ugrizy']
flux_err_cols+=[f'mag_{b}_roman_err' for b in 'YJH']
flux_cols

In [ ]:
#_rail_to_lephare_input(train_data_handle,flux_cols,flux_err_cols)

In [ ]:
# Update the default
config=lsst_default_config.copy()
updates={
    "MAG_REF": "2",
    "ERR_SCALE": "0.02",
    "FILTER_CALIB": "0",
    "FILTER_LIST": lsst_default_config["FILTER_LIST"]+",roman/Roman_WFI.F106.dat,roman/Roman_WFI.F129.dat,roman/Roman_WFI.F158.dat",
    "GLB_CONTEXT": '0',
    "MABS_CONTEXT": '0',
    # Remove AGN for speed
    'ZPHOTLIB': 'LSST_STAR_MAG,LSST_GAL_MAG',
    'Z_STEP': '0.02,0.,3.',
    'EM_DISPERSION': '0.5,1.,1.5',
    'ERR_FACTOR': '1.',
    'MOD_EXTINC':"16,24,24,31,24,31,24,31",
}
config.update(updates)
params={f"lephare.{k}": v for k, v in updates.items()}
params['gal.MOD_EXTINC']="16,24,24,31,24,31,24,31"

In [ ]:
params

In [ ]:
config

In [ ]:
lp.data_retrieval.get_auxiliary_data(keymap=config)

In [ ]:
inform_lephare = LephareInformer.make_stage(
    name="inform_lephare",
    nondetect_val=np.nan,
    model="lephare.pkl",
    hdf5_groupname="",
    **params,
    bands=flux_cols,
    err_bands=flux_err_cols,
    ref_band='mag_g_lsst',

    # These overwrite the config ZSTEP so must be set
    zmin=float(config['Z_STEP'].split(',')[1]),
    zmax=float(config['Z_STEP'].split(',')[2]),
    nzbins=1+float(config['Z_STEP'].split(',')[2])/float(config['Z_STEP'].split(',')[0]),
)

In [ ]:
inform_lephare.inform(train_data_handle)

In [ ]:
# means = [[0.3, 0.4], [0.5, 0.5], [0.6, 0.8]]
# stds = [[0.2, 0.4], [0.1, 0.3], [0.05, 0.3]]
# weights = [[0.8, 0.2], [0.7, 0.3], [0.8, 0.2]]
# ens = qp.mixmod.create_ensemble(means=means,stds=stds,weights=weights)
# ensemble.write_to(<output_filename.hdf5>)

In [ ]:
inform_lephare.model

In [ ]:
import pickle
# All 'models' are identical unless we want to perform seperate auto_adapt on the training samples
for taskset_ in TASKSETS:
    for sim_ in SIMS:
        for scenario_ in SCENARIOS:
            key = f"{taskset_}_{sim_}_pz_model_{scenario_}"
            with open(f'../evaluation/pz_challenge_{key}.pkl', "wb") as f:
                pickle.dump(inform_lephare.model, f)

In [ ]:
inform_lephare.model

In [ ]:
all_keys=[]
with open(lp.LEPHAREDIR 
          + '/examples/output.para', 
         # + '/alloutputkeys.txt', 

          "r") as f:
    content = f.read()
for l in content.split('\n'):
    if len(l)>0:
        if l.split()[0][0]!='#':
            all_keys.append(l.split()[0])
all_keys.append('PDF_BAY_ZG()')
",".join(all_keys)

In [ ]:
# Run all combinations - very slow!
for taskset_ in TASKSETS:
    for sim_ in SIMS:
        for scenario_ in SCENARIOS:
            for run_ in ['training','test']:
                key = f"{taskset_}_{sim_}_{run_}_{scenario_}"
                globals()[f"estimate_lephare_{key}"] = LephareEstimator.make_stage(
                    name=f"estimate_{key}",
                    # nondetect_val=np.nan,
                    model="lephare.pkl", #inform_lephare.get_handle("model"),
                    hdf5_groupname="",
                    #aliases=dict(input="test_data", output="lephare_estim"),
                    #use_inform_offsets=False,
                    bands=flux_cols,
                    err_bands=flux_err_cols,
                    output_keys=all_keys,
                    #lephare_config=estimate_config.copy(),
                )
                
                globals()[f"estimated_lephare_{key}"] = globals()[f"estimate_lephare_{key}"].estimate(data_dict[f"{key}"])
                globals()[f"estimated_lephare_{key}"].data.ancil["object_id"] = data_dict[f"{key}"]["object_id"].astype(int)
                outname = f'../evaluation/pz_challenge_{key}.hdf5'.replace('_test_','_pz_estimate_')
                if run_=='training':
                    outname = f'../evaluation/pz_challenge_{taskset_}_{sim_}_pz_evaluation_{scenario_}.hdf5'
                print(outname)
                globals()[f"estimated_lephare_{key}"].path = outname
                globals()[f"estimated_lephare_{key}"].write()
                if run_=='training':
                    tab=Table(globals()[f"estimated_lephare_{key}"].data.ancil)
                    tab['ZSPEC']=data_dict[f"{key}"]['redshift'] # Should this already be propagated?
                    test_utils = lp.PlotUtils(
                            tab,
                            sel_filt=3,
                            #         u  g  r  z  J  K (subbing Roman H for K)
                            pos_filt=[0, 1, 2, 4, 7, 8],
                            range_z=[0, 0.5, 1, 1.5, 3],
                            range_mag=[19, 20.5, 21.5, 22.5, 25],
                        )
                    kwargs = {"zgrid": np.linspace(inform_lephare.config.zmin, inform_lephare.config.zmax, inform_lephare.config.nzbins)}
                    test_utils.save_photoz_plots_pdf(f'./{key}.pdf', **kwargs)

In [ ]:
tab[:5]

In [ ]:
len(np.unique(tab['EBV_BEST']))

In [ ]:
key

In [ ]:
type(globals()[f'estimated_lephare_{key}'])

In [ ]:
# Copy the generated files across to make the tarball in a convenient location

In [ ]:
!mkdir submit_lephare

In [ ]:
#!cp -r ../evaluation/* ./submit_lephare
!cp -r /Users/rshirley/Library/Caches/lephare/runs/inform_lephare/* ../evaluation

In [ ]:
!ls ../evaluation


In [ ]:
#!tar --disable-copyfile -czf submit_lephare.tgz submit_lephare/

In [ ]:
!ls ../evaluation

In [ ]:
# Leave these if you want to run the metrics notebook
!rm ../evaluation/*evaluation*.hdf5

In [ ]:
# We want the top level folder to not be made to handle the paths correctly
!tar --disable-copyfile -czf submit_lephare.tgz -C ../evaluation .

# Upload tarfile

The tar file should then be uploaded and linked in tests/test_lephare.py

Currently at https://www.raphaelshirley.co.uk/data/submit_lephare.tgz

In [ ]:
# Put the training files back for using the metrics notebook
for taskset_ in TASKSETS:
    for sim_ in SIMS:
        for scenario_ in SCENARIOS:
            for run_ in ['training']:
                key = f"{taskset_}_{sim_}_{run_}_{scenario_}"
                # globals()[f"estimate_lephare_{key}"] = LephareEstimator.make_stage(
                #     name=f"estimate_{key}",
                #     # nondetect_val=np.nan,
                #     model="lephare.pkl", #inform_lephare.get_handle("model"),
                #     hdf5_groupname="",
                #     #aliases=dict(input="test_data", output="lephare_estim"),
                #     #use_inform_offsets=False,
                #     bands=flux_cols,
                #     err_bands=flux_err_cols,
                #     output_keys=all_keys,
                #     #lephare_config=estimate_config.copy(),
                # )
                
                # globals()[f"estimated_lephare_{key}"] = globals()[f"estimate_lephare_{key}"].estimate(data_dict[f"{key}"])
                # globals()[f"estimated_lephare_{key}"].data.ancil["object_id"] = data_dict[f"{key}"]["object_id"].astype(int)
                outname = f'../evaluation/pz_challenge_{key}.hdf5'.replace('_test_','_pz_estimate_')
                if run_=='training':
                    outname = f'../evaluation/pz_challenge_{taskset_}_{sim_}_pz_evaluation_{scenario_}.hdf5'
                print(outname)
                globals()[f"estimated_lephare_{key}"].path = outname
                globals()[f"estimated_lephare_{key}"].write()
                # if run_=='training':
                #     tab=Table(globals()[f"estimated_lephare_{key}"].data.ancil)
                #     tab['ZSPEC']=data_dict[f"{key}"]['redshift'] # Should this already be propagated?
                #     test_utils = lp.PlotUtils(
                #             tab,
                #             sel_filt=3,
                #             #         u  g  r  z  J  K (subbing Roman H for K)
                #             pos_filt=[0, 1, 2, 4, 7, 8],
                #             range_z=[0, 0.5, 1, 1.5, 3],
                #             range_mag=[19, 20.5, 21.5, 22.5, 25],
                #         )
                #     kwargs = {"zgrid": np.linspace(inform_lephare.config.zmin, inform_lephare.config.zmax, inform_lephare.config.nzbins)}
                #     test_utils.save_photoz_plots_pdf(f'./{key}.pdf', **kwargs)